In [ ]:
from pathlib import Path

In [ ]:
import pandas as pd

from run_utils import (
    prepare_data, build_cvae_lstm_model, train_model,
    save_artifacts, load_artifacts,
    generate_timespan, generate_n_weeks,
)

In [ ]:
# SHARED_DIR =

In [ ]:
TRAINING_DATA_PATH = SHARED_DIR_PATH / "training_data" / "greg_tmp"
ARTIFACTS_DIR_PATH = SHARED_DIR_PATH / "artifacts"
RUN_DIR_PATH  = ARTIFACTS_DIR_PATH / "run_1"
WEIGHTS_PATH  = RUN_DIR_PATH / "models_weights"
MODEL_PATH  = WEIGHTS_PATH / "cvae_lstm_v2_0.weights.h5"

In [ ]:
CVAE_LSTM_CONFIG = dict(
    latent_dim=32,
    hidden_dim=256,
    n_layers=2,
    use_attention=True,
    n_heads=4,
    beta=0.0,
    learning_rate=5e-4,
    epochs = 3,
    batch_size = 64,
    target_beta = 1.0,
    anneal_epochs = 20,
    lr_patience = 15,
    early_stop_patience = 30,
    min_lr = 1e-5
)

In [ ]:
model, data = load_artifacts(
    out_dir = RUN_DIR_PATH,
    weights_path = MODEL_PATH,
    latent_dim = CVAE_LSTM_CONFIG["latent_dim"],
    hidden_dim = CVAE_LSTM_CONFIG["hidden_dim"],
    n_layers = CVAE_LSTM_CONFIG["n_layers"],
    use_attention = CVAE_LSTM_CONFIG["use_attention"],
    n_heads = CVAE_LSTM_CONFIG["n_heads"],
    beta = CVAE_LSTM_CONFIG["beta"],
    learning_rate = CVAE_LSTM_CONFIG["learning_rate"],
    )

In [ ]:
# cell_id = ..

In [ ]:
df = generate_timespan(model=model, **data,
    cell_id= .., date_start="2023-12-12", date_end="2024-03-12", seed=42)

In [ ]:
df["distname"] = df["cell_id"]
df.drop("cell_id",axis=1, inplace=True)

In [ ]:
long_df = df.drop(["window_anchor", "seed"], axis=1).melt(
    id_vars=["distname", "timestamp"],
    var_name="kpi_id",
    value_name="kpi_value"
)
long_df

In [ ]:
TRAINING_DATASET_PATH = SHARED_DIR_PATH / "training_data" / "greg_tmp"
pdf = pd.read_parquet(TRAINING_DATASET_PATH / "wide_win_idx")
pdf[(pdf["distname"] == ..) & (pdf["window_anchor"] == '2023-12-12 00:00:00')]

In [ ]:
pdf["timestamp"] = pd.to_datetime(pdf["window_anchor"]) + pd.to_timedelta(pdf["hour_idx"], unit="h")

In [ ]:
long_pdf = pdf.drop(["window_anchor", "bts_id", "hour_idx"], axis=1)[pdf["distname"] == ..].melt(
    id_vars=["distname", "timestamp"],
    var_name="kpi_id",
    value_name="kpi_value"
)
long_pdf

In [ ]:
import pandas as pd
import plotly.express as px

def plot(df, chosen_kpi, chosen_distname):
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df_kpi = df[df["kpi_id"] == chosen_kpi].sort_values("timestamp")

    fig = px.line(
            df_kpi,
            x="timestamp",
            y="kpi_value",
            color="distname",
            title=f"KPI {chosen_kpi} over time for distname {chosen_distname}"
        )

    fig.update_layout(template="plotly_white")
    fig.show()